## Model Performance Metrics

In [ ]:
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from pathlib import Path

# Add project root to path
project_root = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.project_utils import setup_project_path
setup_project_path()

from src.utils.model_loader import load_model
from src.pipeline.datasets import ProductDataset
from torch.utils.data import DataLoader


### Load Model, Data, and Labels

In [ ]:
# Load model and label mapping using correct paths
model_path = str(project_root / "models/best_model.pth")
label_mapping_path = str(project_root / "src/data/dataset/label_mapping.json")
model, label_mapping = load_model(model_path, label_mapping_path)
model.eval()

# Load test/validation data using correct path
df = pd.read_csv(str(project_root / "src/data/dataset/final_cnn_training_data.csv"))
image_paths = df['image_path'].tolist()
labels = df['label'].astype(int).tolist()

# Prepare dataset and dataloader with project_root for path resolution
dataset = ProductDataset(image_paths, labels, project_root=project_root)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

### Get Predictions and True Labels

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels_batch in loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

### Accuracy and F1 Score

In [ ]:
acc = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average='macro')
f1_micro = f1_score(all_labels, all_preds, average='micro')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"Accuracy: {acc:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Micro: {f1_micro:.4f}")
print(f"F1 Weighted: {f1_weighted:.4f}")

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[label_mapping[str(i)] for i in range(len(label_mapping))],
            yticklabels=[label_mapping[str(i)] for i in range(len(label_mapping))])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

### Classification Report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=[label_mapping[str(i)] for i in range(len(label_mapping))]))

### ROC Curve for Each Class

In [ ]:
from sklearn.preprocessing import label_binarize

# Binarize the labels for ROC AUC (One-vs-Rest)
n_classes = len(label_mapping)
y_true_bin = label_binarize(all_labels, classes=list(range(n_classes)))
y_pred_bin = label_binarize(all_preds, classes=list(range(n_classes)))

# Compute ROC AUC for each class
for i in range(n_classes):
    try:
        auc = roc_auc_score(y_true_bin[:, i], y_pred_bin[:, i])
        print(f"Class {i} ({label_mapping[str(i)]}): ROC AUC = {auc:.4f}")
    except ValueError:
        print(f"Class {i} ({label_mapping[str(i)]}): ROC AUC not defined (only one class present)")